# Eigen-Interactions Filtering

Apply the eigen-interactions package to sequences from the joint library (~57k).

Steps:
1. Load sequences from joint library
2. Screen predictions → select well-predicted, high-activity sequences
3. Compute DeepLIFT/SHAP attributions across cell types
4. Eigendecompose cross-cell-type importance
5. Annotate motifs and cross-reference TF expression
6. Steer sequences along eigenvectors

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

REPO = '/grid/wsbs/home_norepl/pmantill/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra'
EIGEN_DIR = os.path.join(REPO, 'eigen-interactions')
sys.path.insert(0, EIGEN_DIR)

import eigen_steering; importlib.reload(eigen_steering)
from eigen_steering import EigenMap, PROMOTER_SEQ, RAND_BARCODE

# Override paths to point at our local models instead of the training repo
eigen_steering.WEIGHTS_PATH = '/grid/wsbs/home_norepl/pmantill/LentiMPRA_mcs/alphagenome_torch_MPRAMoCon/weights/model_fold_0.safetensors'
eigen_steering.RESULTS_DIR = os.path.join(REPO, 'models')

# Our model dirs have best_stage2.pt directly (no checkpoints/ subdir),
# so patch _load_model to look in the right place
_orig_load = EigenMap._load_model
def _patched_load(self, ct, squeeze=False):
    model_name = self.model_names[ct]
    ckpt_dir = os.path.join(eigen_steering.RESULTS_DIR, model_name)
    # Check both layouts: direct and checkpoints/
    direct = os.path.join(ckpt_dir, 'best_stage2.pt')
    nested = os.path.join(ckpt_dir, 'checkpoints', 'best_stage2.pt')
    if os.path.exists(direct) and not os.path.exists(nested):
        os.makedirs(os.path.join(ckpt_dir, 'checkpoints'), exist_ok=True)
        os.symlink(direct, nested)
    return _orig_load(self, ct, squeeze=squeeze)
EigenMap._load_model = _patched_load

DATA_CSV = os.path.join(REPO, 'data', 'joint_library_combined.csv')
print(f'Eigen-interactions loaded from {EIGEN_DIR}')

In [ ]:
# Load joint library
df = pd.read_csv(DATA_CSV)
df = df.dropna(subset=['sequence', 'K562_log2FC', 'HepG2_log2FC', 'WTC11_log2FC']).reset_index(drop=True)
print(f'{len(df)} sequences with K562 + HepG2 + WTC11 data')

# Best models per cell line by Pearson R on joint library (validate_models.ipynb)
# K562:  do075 → r=0.8915 | do06 → r=0.8837 | do03 → r=0.8683
# HepG2: do03  → r=0.8750 | do075 → r=0.8703 | do06 → r=0.8665
# WTC11: do075 → r=0.8457 | do06 → r=0.8371 | do03 → r=0.8262
MODEL_NAMES = {
    'K562':  'K562_v6_do075',
    'HepG2': 'HepG2_v6_do03',
    'WTC11': 'WTC11_v6_do075',
}